In [1]:
import pandas as pd

from dvf.config import settings

df = pd.read_csv(
    settings.raw_data_dir / "dvf_33_2024.csv.gz",
    compression= "gzip",
    low_memory= False,
)

print(df.shape)
df.head

(82927, 40)


<bound method NDFrame.head of        id_mutation date_mutation  numero_disposition nature_mutation  \
0      2024-363954    2024-01-03                   1           Vente   
1      2024-363955    2024-01-08                   1           Vente   
2      2024-363956    2024-01-05                   1           Vente   
3      2024-363956    2024-01-05                   1           Vente   
4      2024-363957    2024-01-08                   1           Vente   
...            ...           ...                 ...             ...   
82922  2024-393381    2024-03-08                   1           Vente   
82923  2024-393382    2024-12-05                   1    Adjudication   
82924  2024-393382    2024-12-05                   1    Adjudication   
82925  2024-393382    2024-12-05                   1    Adjudication   
82926  2024-393382    2024-12-05                   1    Adjudication   

       valeur_fonciere  adresse_numero adresse_suffixe  \
0             336120.0            16.0         

In [2]:
df["nature_mutation"].value_counts()

nature_mutation
Vente                                 76020
Vente en l'état futur d'achèvement     5662
Echange                                 957
Adjudication                            178
Vente terrain à bâtir                    92
Expropriation                            18
Name: count, dtype: int64

Quelles nature_mutation écarter ?
Le critère n'est pas la fréquence mais le sens : ce prix reflète-t-il un marché libre ? On garde Vente. On écarte Adjudication (enchère, souvent forcée), Expropriation (prix imposé), Échange (pas d'argent), Vente terrain à bâtir (du terrain, pas du bâti). Vente en l'état futur d'achèvement : à trancher explicitement.

In [3]:
print("lignes :", len(df))
print("Mutations uniques :", df["id_mutation"].nunique())
print("Valeur fonciere manquante:", df["valeur_fonciere"].isna().mean().round(3))
df["valeur_fonciere"].describe()

lignes : 82927
Mutations uniques : 29429
Valeur fonciere manquante: 0.007


count    8.237100e+04
mean     8.959978e+05
std      3.768020e+06
min      1.000000e+00
25%      1.060000e+05
50%      2.219500e+05
75%      3.946750e+05
max      6.860122e+07
Name: valeur_fonciere, dtype: float64

plus de lignes que de mutations uniques ?
Ce ne sont pas des doublons. Une mutation = une vente, et une vente peut porter sur plusieurs biens (appartement + parking + cave). DVF émet une ligne par lot mais recopie le prix total sur chacune. drop_duplicates() ne supprimerait rien, car les lignes diffèrent. Correction : agréger par id_mutation — sommer les surfaces, compter les lots, garder un seul prix.

Où couper les valeurs extrêmes ?
Pas avec l'écart-type : les prix sont fortement asymétriques à droite, et moyenne comme écart-type sont eux-mêmes déformés par les extrêmes. Deux options défendables : les quantiles (1ᵉ–99ᵉ percentile), ou une borne métier justifiable à l'oral (« sous 10 000 € ce n'est pas une transaction réelle, au-dessus de 5 M€ c'est hors périmètre »). Une partie des extrêmes disparaîtra d'elle-même une fois Q1 corrigé.

Que faire des 0,7 % de valeurs manquantes ?
Les supprimer. Et la règle derrière : on n'impute jamais la variable cible — inventer un prix, c'est demander au modèle d'apprendre ton invention. L'imputation ne concerne que les variables explicatives.

In [4]:
compte = df["id_mutation"].value_counts()
exemple = compte[compte > 1].index[0]

df[df["id_mutation"] == exemple][[
    "id_mutation", "date_mutation", "valeur_fonciere",
    "type_local", "surface_reelle_bati", "nombre_pieces_principales",
    "adresse_nom_voie",
]]

,id_mutation,date_mutation,valeur_fonciere,type_local,surface_reelle_bati,nombre_pieces_principales,adresse_nom_voie
81304,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81305,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81306,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81307,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81308,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
...,...,...,...,...,...,...,...
81714,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81715,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81716,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
81717,2024-393059,2024-12-19,25122574.0,NaN,NaN,NaN,FONTBELLEAU
